## CSI 4142 Assignment 4

Lucas Gavura, 300310069

Jonathan Cojita, 300283917

April 7, 2026

Work Split:

Lucas Gavura: Part 1, studies 1 and 3

Jonathan Cojita: Part 2, 

Database summary:

Database 1: The Movies Dataset

 Columns

 Rows

Database 2: 

Columns

Rows

### Movies Dataset - Data Preparation


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings
import kagglehub
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 60)

print('Libraries imported successfully.')

In [ ]:
#Import dataset and preview
path = kagglehub.dataset_download(
    "rounakbanik/the-movies-dataset"
)

df_raw = pd.read_csv(f"{path}/movies_metadata.csv:")

#Preview the dataset and info about the columns and data types
print(df_raw.head())
print(df_raw.info())

In [ ]:
# Data types and non-null counts 
print('Data Types & Non-Null Counts')
df_raw.info()

In [ ]:
# Missing value analysis 
missing = df_raw.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('=== Missing Values ===')
print(missing_df)

In [ ]:
# Visualisation of the missing values bar chart
fig, ax = plt.subplots(figsize=(10, 5))
missing_df['Missing %'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Missing Values per Column (%)', fontsize=14)
ax.set_ylabel('Missing (%)')
ax.set_xlabel('Column')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df = df_raw.copy()

# Convert to numeric
for col in ['budget', 'revenue', 'runtime']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['vote_average', 'vote_count', 'popularity']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Parse genres from JSON to list
def parse_genres(val):
    try:
        parsed = ast.literal_eval(val)
        return [g['name'] for g in parsed if isinstance(g, dict) and 'name' in g]
    except:
        return []

df['genres_list'] = df['genres'].apply(parse_genres)
df['genres_str'] = df['genres_list'].apply(lambda x: ' '.join(x) if x else '')

# Drop rows
df = df.dropna(subset=['title'])
df = df[df['genres_list'].map(len) > 0]

# Replace zeros
df['budget'] = df['budget'].replace(0, np.nan)
df['revenue'] = df['revenue'].replace(0, np.nan)
df['runtime'] = df['runtime'].replace(0, np.nan)

# Drop rows missing runtime
df = df.dropna(subset=['runtime'])

# Keep movies with meaningful vote amt
df = df[df['vote_count'] >= 10]

df = df.reset_index(drop=True)
print(f'Cleaned dataset shape: {df.shape}')
df[['title', 'genres_list', 'budget', 'revenue', 'runtime', 'vote_average', 'popularity']].head(5)

In [ ]:
# Visualisation of Runtime distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(df['runtime'].dropna(), bins=60, color='darkorange', edgecolor='black', alpha=0.8)
ax.set_title('Distribution of Movie Runtime (minutes)', fontsize=13)
ax.set_xlabel('Runtime (min)')
ax.set_ylabel('Count')
ax.axvline(df['runtime'].median(), color='navy', linestyle='--', label=f'Median: {df["runtime"].median():.0f} min')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation of genres
from collections import Counter

all_genres = [g for sublist in df['genres_list'] for g in sublist]
genre_counts = Counter(all_genres).most_common(15)
genres_df = pd.DataFrame(genre_counts, columns=['Genre', 'Count'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(genres_df['Genre'][::-1], genres_df['Count'][::-1], color='teal', edgecolor='black')
ax.set_title('Top 15 Genres in the Dataset', fontsize=13)
ax.set_xlabel('Number of Movies')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisation of Budget vs Revenue
df_br = df.dropna(subset=['budget', 'revenue'])
df_br = df_br[(df_br['budget'] > 1e5) & (df_br['revenue'] > 1e5)]  # remove extreme near-zeros

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_br['budget'] / 1e6, df_br['revenue'] / 1e6,
           alpha=0.3, s=10, color='purple')
ax.set_title('Budget vs Revenue (millions USD)', fontsize=13)
ax.set_xlabel('Budget ($M)')
ax.set_ylabel('Revenue ($M)')
ax.set_xlim(0, 400)
ax.set_ylim(0, 2500)
plt.tight_layout()
plt.show()

print(f'Movies with both budget and revenue data: {len(df_br)}')

In [ ]:
# Normalize numerical our columns
scaler = MinMaxScaler()

for col in ['budget', 'revenue', 'runtime', 'popularity', 'vote_average']:
    df[col + '_filled'] = df[col].fillna(df[col].median())

num_cols = ['budget_filled', 'revenue_filled', 'runtime_filled', 'popularity_filled', 'vote_average_filled']
df[num_cols] = scaler.fit_transform(df[num_cols])

print('Normalized numerical columns:')
df[num_cols].describe().round(3)

## Study 1 - Similarity Measures

## Study 3 – Content-Based Recommendation System